# ingest_file_batch\nGenerated from 01_ingestion/ingest_file_batch.py for Databricks notebook import.\n

In [ ]:
\nfrom __future__ import annotations\nfrom pathlib import Path\nimport sys\nsys.path.append(str(Path(__file__).resolve().parent.parent / "00_common"))\nfrom attachment_utils import handle_attachment_file\n"""Batch file ingestion adapter for full and incremental loads.\n\nThis adapter is designed for Bronze RAW ingestion where heterogeneous files\n(JSON/JSONL/mixed) can be loaded as raw payload using ``binaryFile``.\n"""\n\n# Allowed source_format values that map to a Spark reader format.\n_FORMAT_MAP = {\n    "json": "json",\n    "jsonl": "json",           # JSONL is newline-delimited JSON — same Spark reader.\n    "binaryFile": "binaryFile",\n    "json_jsonl_mixed": "binaryFile",  # Mixed: read raw bytes, parse later.\n    "csv": "csv",\n    "parquet": "parquet",\n    "avro": "avro",\n    "orc": "orc",\n    "text": "text",\n}\n\n\ndef _reader_format(source: dict, global_config: dict) -> str:\n    source_format = str(source.get("source_format", "")).strip()\n    file_defaults = global_config.get("connections", {}).get("file_defaults", {})\n    default_format = str(file_defaults.get("format", "binaryFile")).strip()\n    return _FORMAT_MAP.get(source_format, default_format)\n\n\ndef ingest_file_batch(spark, source: dict, global_config: dict, source_options: dict | None = None):\n    """Read files from *source_path* as a Spark batch DataFrame.\n\n    Parameters\n    ----------\n    spark:\n        Active SparkSession.\n    source:\n        Source registry row.  Must contain ``source_path``.\n    global_config:\n        Parsed global config dict.\n    source_options:\n        Parsed ``source_options_json`` dict.  All scalar values are forwarded\n        as Spark reader options.  Special keys:\n\n        - ``incremental_start_timestamp`` — ISO-8601 string; rows whose\n          ``modificationTime`` is older are filtered out (binaryFile only).\n        - ``fail_on_empty`` — ``"true"`` (default) raises ``ValueError`` if the\n          path returns zero rows, catching path/config mismatches early.\n        - Any other key is forwarded directly to the Spark reader as a string.\n\n    Returns\n    -------\n    pyspark.sql.DataFrame\n    """\n    source_path = source.get("source_path", "")\n    if not source_path:\n        raise ValueError("source_path is required for FILE ingestion")\n\n    source_options = source_options or {}\n    fmt = _reader_format(source, global_config)\n    reader = spark.read.format(fmt)\n\n    # Reserved keys that are handled explicitly — not forwarded to the reader.\n    _reserved = {"incremental_start_timestamp", "fail_on_empty", "file_ingest_mode"}\n\n    for k, v in source_options.items():\n        if k in _reserved:\n            continue\n        if isinstance(v, (str, int, float, bool)):\n            reader = reader.option(k, str(v))\n\n    df = reader.load(source_path)\n\n    # Handle images and attachments if fields are specified in source_options\n    image_field = source_options.get("image_field")\n    attachment_field = source_options.get("attachment_field")\n    dest_dir = source_options.get("attachment_dest_dir")\n    if image_field and image_field in df.columns and dest_dir:\n        # Copy all images referenced in the image_field column to dest_dir\n        image_paths = [\n            row[image_field]\n            for row in df.select(image_field).distinct().collect()\n            if row[image_field]\n        ]\n        for img_path in image_paths:\n            try:\n                new_path = handle_attachment_file(img_path, dest_dir)\n                print(f"Copied image {img_path} to {new_path}")\n            except Exception as e:\n                print(f"Failed to copy image {img_path}: {e}")\n    if attachment_field and attachment_field in df.columns and dest_dir:\n        # Copy all attachments referenced in the attachment_field column to dest_dir\n        attach_paths = [row[attachment_field] for row in df.select(attachment_field).distinct().collect() if row[attachment_field]]\n        for att_path in attach_paths:\n            try:\n                new_path = handle_attachment_file(att_path, dest_dir)\n                print(f"Copied attachment {att_path} to {new_path}")\n            except Exception as e:\n                print(f"Failed to copy attachment {att_path}: {e}")\n\n    load_type = str(source.get("load_type", "incremental")).lower()\n    if load_type == "incremental":\n        from pyspark.sql import functions as F\n\n        # Optional runtime filter — keep only files modified after this timestamp.\n        incremental_start = source_options.get("incremental_start_timestamp")\n        if incremental_start and "modificationTime" in df.columns:\n            df = df.filter(F.col("modificationTime") > F.to_timestamp(F.lit(str(incremental_start))))\n\n    # NOTE: empty-dataset detection is intentionally NOT done here.\n    # Calling df.rdd.isEmpty() or df.count() at ingestion time would trigger a\n    # full Spark action before the orchestrator's own raw_count = raw_df.count().\n    # That would scan the data twice.  The orchestrator handles empty detection\n    # by checking raw_count == 0 after ingestion.\n    return df\n